# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/innouguru/flyrank-intenship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding #2 — The Content Performance Curve

**Paper finding:**  
The paper observes that content health peaks around 61–90 days, declines substantially around 271–365 days, and shows a higher health score among some 365+ day content that was refreshed. :contentReference[oaicite:0]{index=0}

**My methodology question:**  
Because this is an observational comparison across content-age buckets, how much of the observed performance difference is associated with content age itself versus differences in the types of pages represented in each age group?

---

### Finding #4 — The Freshness Multiplier

**Paper finding:**  
The paper reports that 365+ day content refreshed within 30 days showed a 3.2× health increase and 57× more impressions, while also noting that the 361+ freshness bucket is small and unstable. :contentReference[oaicite:1]{index=1}

**My methodology question:**  
How were the refreshed and unrefreshed pages selected, and does the comparison account for pre-existing differences between pages that were chosen for refresh and those that were not?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
from google.colab import userdata

# Retrieve the Hugging Face token stored in Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Check whether the token was successfully loaded
print("Token loaded:", HF_TOKEN is not None)


Token loaded: True


In [2]:
import duckdb        # Import DuckDB for working with data using SQL

con = duckdb.connect()      # Create an in-memory DuckDB connection

# Create a Hugging Face secret in DuckDB
con.execute(
    f"""CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )"""
)

rel = "hf://datasets/FlyRank/internship-warehouse"

In [3]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier

### Data setup

This notebook is a separate validation audit of the Week-5 model, so it must be reproducible independently of the Week-5 notebook.

I recreate the data needed for the audit from the FlyRank internship warehouse using the same data source and core data-availability rules used in Week 5.

The analysis focuses on the temporal relationship between current-month candidate features and the following month's observed outcome.

### Recreating the Week-5 candidate definition

The Week-5 model was trained only on pages identified as CTR candidates by the Week-4 rule.

I reproduce that candidate definition here so that the Week-6 audit evaluates the same modeling problem rather than changing the task.

A candidate is a page whose current-month CTR is below the median CTR of its position peers within the same client and month. I also retain only position-peer groups with at least 15 observations, consistent with the Week-5 setup.

In [4]:
# Build the monthly historical feature dataset used for model development.
# October 2025 through February 2026 are included so that we can create
# the training period (October-January) and validation period (February).

training_data = con.sql(
    f"""
    SELECT
        DATE_TRUNC('month', report_date) AS month,
        client_hash_id,
        content_hash_id,

        -- Total monthly search impressions.
        SUM(gsc_impressions) AS gsc_impressions,

        -- Total monthly search clicks.
        SUM(gsc_clicks) AS gsc_clicks,

        -- Impression-weighted average search position.
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_avg_position * gsc_impressions)
                 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_avg_position,

        -- Total monthly pageviews.
        SUM(ga4_pageviews) AS ga4_pageviews,

        -- Total monthly engaged sessions.
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    -- Use only months that will be used to construct
    -- the Random Forest training examples.
    WHERE report_date >= '2025-10-01'
      AND report_date < '2026-03-01'

      -- Match the Week 4 data availability requirement.
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE

    GROUP BY
        DATE_TRUNC('month', report_date),
        client_hash_id,
        content_hash_id
    ORDER BY
        month,
        client_hash_id,
        content_hash_id
    """
).df()

# Check how much data was extracted.
print("Training rows:", len(training_data))

# Check the months included in the training data.
print("\nMonths:")
print(training_data["month"].sort_values().unique())

# Check the available columns.
print("\nColumns:")
print(training_data.columns.tolist())

# Display a few records.
training_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training rows: 74213

Months:
<DatetimeArray>
['2025-10-01 00:00:00', '2025-11-01 00:00:00', '2025-12-01 00:00:00',
 '2026-01-01 00:00:00', '2026-02-01 00:00:00']
Length: 5, dtype: datetime64[us]

Columns:
['month', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


,month,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,2025-10-01,client_23a62021009f63c4,content_0013b0a02c45a7c8,90.0,0.0,4.000000,4.0,0.0
1,2025-10-01,client_23a62021009f63c4,content_0017e6d53661a061,69.0,0.0,5.492754,2.0,0.0
2,2025-10-01,client_23a62021009f63c4,content_01063b6675ed287a,48.0,1.0,4.604167,4.0,0.0
3,2025-10-01,client_23a62021009f63c4,content_015ff2d7df79201c,93.0,3.0,7.333333,8.0,2.0
4,2025-10-01,client_23a62021009f63c4,content_0161abfca8c5a51c,4.0,0.0,5.000000,2.0,0.0


In [5]:
# Keep only observations with a valid search position.
training_data = training_data[
    training_data["gsc_avg_position"] >= 1
].copy()

# Calculate monthly CTR.
training_data["ctr"] = (
    training_data["gsc_clicks"] * 100.0
    / training_data["gsc_impressions"]
)

# Create 10-position peer groups.
training_data["position_start"] = (
    ((training_data["gsc_avg_position"] - 1) // 10) * 10 + 1
)

# Calculate the median CTR within each client, month,
# and position peer group.
training_data["peer_median_ctr"] = (
    training_data
    .groupby(
        ["month", "client_hash_id", "position_start"]
    )["ctr"]
    .transform("median")
)

# Count observations in each peer group.
training_data["peer_count"] = (
    training_data
    .groupby(
        ["month", "client_hash_id", "position_start"]
    )["content_hash_id"]
    .transform("count")
)

# Keep only peer groups with at least 15 observations.
training_data = training_data[
    training_data["peer_count"] >= 15
].copy()

# Reproduce the Week-4 candidate rule.
training_data["ctr_below_position_peers"] = (
    training_data["ctr"]
    < training_data["peer_median_ctr"]
)

print("Training observations:", len(training_data))
print(
    "CTR candidates:",
    training_data["ctr_below_position_peers"].sum()
)

Training observations: 70671
CTR candidates: 28486


### Constructing the future outcome

The target represents the outcome observed in the month following the prediction month.

For example:

- October features → November outcome
- January features → February outcome
- February features → March outcome

The future-month CTR and peer median are used only to construct the target. They are not included among the model features.

This distinction is important for the leakage audit: information from the future is allowed to define the outcome being predicted, but it must not be available as an input feature at prediction time.

In [6]:
# Build one monthly observation for each content page.
#
# The warehouse contains daily observations, but our model
# operates at the page-month level.

future_monthly = con.sql(
    f"""
    WITH daily AS (
        SELECT
            month,
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position

        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )

        -- Use the months immediately after our five
        -- training months.
        WHERE month >= '2025-11'
          AND month <= '2026-03'

          -- Use the same data-availability requirement as Week 4.
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE

          -- Exclude observations without a valid search position.
          AND gsc_avg_position >= 1
    ),

    monthly_pages AS (
        SELECT
            month,
            client_hash_id,
            content_hash_id,

            -- Aggregate clicks and impressions across the month.
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,

            -- Calculate the average search position for the month.
            AVG(gsc_avg_position) AS avg_position

        FROM daily

        GROUP BY
            month,
            client_hash_id,
            content_hash_id
    ),

    page_metrics AS (
        SELECT
            month,
            client_hash_id,
            content_hash_id,
            avg_position,

            -- Calculate monthly CTR from total clicks
            -- divided by total impressions.
            CASE
                WHEN impressions > 0
                THEN (clicks * 100.0) / impressions
                ELSE NULL
            END AS ctr,

            -- Create the same 10-position peer groups
            -- used in the Week 4 rule.
            FLOOR((avg_position - 1) / 10) * 10 + 1
                AS position_start

        FROM monthly_pages
    ),

    peer_stats AS (
        SELECT
            month,
            position_start,

            -- Calculate the median CTR for each
            -- month and position peer group.
            MEDIAN(ctr) AS peer_median_ctr,

            -- Count the number of pages in each peer group.
            COUNT(*) AS peer_count

        FROM page_metrics

        WHERE ctr IS NOT NULL

        GROUP BY
            month,
            position_start
    )

    SELECT
        p.month,
        p.client_hash_id,
        p.content_hash_id,
        p.avg_position AS gsc_avg_position,
        p.ctr,
        p.position_start,
        s.peer_median_ctr,
        s.peer_count

    FROM page_metrics p

    -- Attach the peer statistics for the same month
    -- and the same position group.
    INNER JOIN peer_stats s
        ON p.month = s.month
        AND p.position_start = s.position_start

    -- Keep only peer groups with at least 15 observations.
    WHERE s.peer_count >= 15
      AND p.ctr IS NOT NULL

    ORDER BY
        p.month,
        p.client_hash_id,
        p.content_hash_id
    """
).df()


# Verify that each page appears only once per month.
duplicate_check = (
    future_monthly
    .groupby(
        ["month", "client_hash_id", "content_hash_id"]
    )
    .size()
)

print(
    "Maximum observations per page-month:",
    duplicate_check.max()
)

print(
    "Future monthly observations:",
    len(future_monthly)
)

print(
    "Months:",
    future_monthly["month"].unique()
)

future_monthly.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Maximum observations per page-month: 1
Future monthly observations: 130091
Months: ['2025-11' '2025-12' '2026-01' '2026-02' '2026-03']


,month,client_hash_id,content_hash_id,gsc_avg_position,ctr,position_start,peer_median_ctr,peer_count
0,2025-11,client_23a62021009f63c4,content_0013b0a02c45a7c8,25.345238,0.000000,21.0,0.000000,662
1,2025-11,client_23a62021009f63c4,content_0017e6d53661a061,12.781149,0.130208,11.0,0.501463,2322
2,2025-11,client_23a62021009f63c4,content_0018b50e392faeb8,7.925000,0.000000,1.0,0.717489,10573
3,2025-11,client_23a62021009f63c4,content_001ccd954baa493f,20.625000,0.000000,11.0,0.501463,2322
4,2025-11,client_23a62021009f63c4,content_002ab5d74a38ddd7,7.791667,0.000000,1.0,0.717489,10573


### Recreating the Week-5 prediction target

I reproduce the Week-5 target construction so that the Week-6 audit evaluates the same modeling problem.

Each observation represents a current-month CTR candidate. Its target is determined by the following month's outcome: the page is labeled positive when its next-month CTR reaches or exceeds the next month's position-peer median.

The future-month variables are used only to construct the target and are not used as model features.

In [7]:
import pandas as pd
# Make copies so we don't accidentally modify the original
# training and future datasets.

current_data = training_data.copy()
next_data = future_monthly.copy()

# Convert the month columns to datetime so we can calculate
# the following month reliably.

current_data["month"] = pd.to_datetime(current_data["month"])
next_data["month"] = pd.to_datetime(next_data["month"])

# Create the month in which we expect to observe the outcome.
#
# Example:
# October 2025 -> November 2025
# November 2025 -> December 2025

current_data["next_month"] = (
    current_data["month"] + pd.DateOffset(months=1)
)

# Keep only pages that were identified as CTR problems
# by the Week 4 rule.
#
# This means the Random Forest will learn which candidates
# are more likely to recover, rather than simply learning
# how to identify low-CTR pages.

current_candidates = current_data[
    current_data["ctr_below_position_peers"]
].copy()

print(
    "Current-month CTR candidates:",
    len(current_candidates)
)

# Select only the columns needed from the next-month data.
next_outcomes = next_data[
    [
        "month",
        "client_hash_id",
        "content_hash_id",
        "ctr",
        "peer_median_ctr",
        "peer_count"
    ]
].copy()

# Rename the outcome columns so it is clear that they belong
# to the following month.

next_outcomes = next_outcomes.rename(
    columns={
        "month": "next_month",
        "ctr": "next_ctr",
        "peer_median_ctr": "next_peer_median_ctr",
        "peer_count": "next_peer_count"
    }
)

# Match each current-month candidate to the same page in
# the following month.

target_data = current_candidates.merge(
    next_outcomes,
    on=[
        "next_month",
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

# Define recovery:
#
# The page was below its position peers in the current month,
# and in the next month its CTR reached or exceeded the
# next month's position-peer median.

target_data["future_ctr_improved"] = (
    target_data["next_ctr"]
    >= target_data["next_peer_median_ctr"]
)

# Convert the Boolean target to 0/1 for model training.

target_data["target"] = (
    target_data["future_ctr_improved"]
    .astype(int)
)

# Inspect the target distribution.

print(
    "Candidate observations with a future outcome:",
    len(target_data)
)

print(
    "Recovered pages:",
    target_data["target"].sum()
)

print(
    "Recovery rate:",
    round(target_data["target"].mean() * 100, 2),
    "%"
)

# Show the first few examples so we can manually verify
# that the target makes sense.

target_data[
    [
        "month",
        "client_hash_id",
        "content_hash_id",
        "ctr",
        "peer_median_ctr",
        "next_month",
        "next_ctr",
        "next_peer_median_ctr",
        "target"
    ]
].head(10)

Current-month CTR candidates: 28486
Candidate observations with a future outcome: 23388
Recovered pages: 10420
Recovery rate: 44.55 %


,month,client_hash_id,content_hash_id,ctr,peer_median_ctr,next_month,next_ctr,next_peer_median_ctr,target
0,2025-10-01,client_23a62021009f63c4,content_0013b0a02c45a7c8,0.0,0.416667,2025-11-01,0.000000,0.000000,1
1,2025-10-01,client_23a62021009f63c4,content_0017e6d53661a061,0.0,0.416667,2025-11-01,0.130208,0.501463,0
2,2025-10-01,client_23a62021009f63c4,content_0161abfca8c5a51c,0.0,0.416667,2025-11-01,0.000000,0.717489,0
3,2025-10-01,client_23a62021009f63c4,content_01d5ec5c852ec2d4,0.0,0.416667,2025-11-01,0.863309,0.717489,1
4,2025-10-01,client_23a62021009f63c4,content_01ffba18790582bf,0.0,0.416667,2025-11-01,0.000000,0.717489,0
5,2025-10-01,client_23a62021009f63c4,content_0210eeea3a738a43,0.0,0.416667,2025-11-01,1.176471,0.501463,1
6,2025-10-01,client_23a62021009f63c4,content_02474d1ff0eca1e5,0.0,0.416667,2025-11-01,0.975610,0.501463,1
7,2025-10-01,client_23a62021009f63c4,content_02642e433b8d6505,0.0,0.416667,2025-11-01,0.126582,0.717489,0
8,2025-10-01,client_23a62021009f63c4,content_02ae546acf5ccffe,0.0,0.416667,2025-11-01,0.366300,0.717489,0
9,2025-10-01,client_23a62021009f63c4,content_02b34cb41e018d75,0.0,0.416667,2025-11-01,0.540541,0.501463,1


### Time-aware validation split

The Week-5 model uses current-month signals to predict whether a CTR candidate will recover in the following month. I therefore preserve the chronological order of the observations when creating the validation split.

I use October–December 2025 as the development period, January 2026 as the validation period for model selection, and February 2026 as the final prediction period.

The February observations contain current-month features, while their targets are determined by the observed March outcomes. The February period is therefore kept out of model selection and used only for the final out-of-time evaluation.

This setup is intended to better represent the decision-support scenario in which a model is trained on historical information, selected using an earlier period, and then applied to a later period that was not used during model selection.

In [8]:
# Define the development period.
# These are the earliest prediction months and will be used
# to train the candidate Random Forest models.
development_df = target_data[
    target_data["month"] < pd.Timestamp("2026-01-01")
].copy()

# Define January 2026 as the validation period.
# This period is used to compare model configurations and
# select the configuration before the final evaluation.
validation_df = target_data[
    target_data["month"] == pd.Timestamp("2026-01-01")
].copy()

# Define February 2026 as the final prediction period.
# Its target represents the outcome observed in March 2026.
# This period must not influence model selection.
final_holdout_df = target_data[
    target_data["month"] == pd.Timestamp("2026-02-01")
].copy()

# Verify the sizes and temporal roles of the three datasets.
print("Development observations:", len(development_df))
print("Validation observations:", len(validation_df))
print("Final holdout observations:", len(final_holdout_df))

print("\nDevelopment months:")
print(
    sorted(
        development_df["month"]
        .dt.strftime("%Y-%m")
        .unique()
    )
)

print("\nValidation month:")
print(
    sorted(
        validation_df["month"]
        .dt.strftime("%Y-%m")
        .unique()
    )
)

print("\nFinal holdout month:")
print(
    sorted(
        final_holdout_df["month"]
        .dt.strftime("%Y-%m")
        .unique()
    )
)

Development observations: 11858
Validation observations: 5795
Final holdout observations: 5735

Development months:
['2025-10', '2025-11', '2025-12']

Validation month:
['2026-01']

Final holdout month:
['2026-02']


### Preparing the model features

I use the same five input features as the Week-5 Random Forest model:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_engaged_sessions`

These features describe the page during the prediction month.

I intentionally exclude the future-outcome fields, including `next_ctr`, `next_peer_median_ctr`, `next_peer_count`, `future_ctr_improved`, and `target`, from the feature matrix.

The target is retained separately as the outcome to be predicted.

In [9]:
# Define the five features used by the Week-5 Random Forest.
# Keeping the same feature set makes the Week-6 comparison
# a validation-design comparison rather than a feature change.

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

# Create the feature matrices and target vectors for each
# temporal period.

X_development = development_df[feature_cols].copy()
y_development = development_df["target"].copy()

X_validation = validation_df[feature_cols].copy()
y_validation = validation_df["target"].copy()

X_final = final_holdout_df[feature_cols].copy()
y_final = final_holdout_df["target"].copy()

# Check the resulting shapes.
print("Development X:", X_development.shape)
print("Development y:", y_development.shape)

print("Validation X:", X_validation.shape)
print("Validation y:", y_validation.shape)

print("Final X:", X_final.shape)
print("Final y:", y_final.shape)

# Confirm that only the intended five features are being used.
print("\nFeatures:")
print(feature_cols)

Development X: (11858, 5)
Development y: (11858,)
Validation X: (5795, 5)
Validation y: (5795,)
Final X: (5735, 5)
Final y: (5735,)

Features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


In [10]:
# Check for missing values in the five model features.
# Missing values could affect model training and must be
# handled consistently before fitting the Random Forest.

print("Missing values in development features:")
print(X_development.isna().sum())

print("\nMissing values in validation features:")
print(X_validation.isna().sum())

print("\nMissing values in final features:")
print(X_final.isna().sum())

Missing values in development features:
gsc_impressions         0
gsc_clicks              0
gsc_avg_position        0
ga4_pageviews           0
ga4_engaged_sessions    0
dtype: int64

Missing values in validation features:
gsc_impressions         0
gsc_clicks              0
gsc_avg_position        0
ga4_pageviews           0
ga4_engaged_sessions    0
dtype: int64

Missing values in final features:
gsc_impressions         0
gsc_clicks              0
gsc_avg_position        0
ga4_pageviews           0
ga4_engaged_sessions    0
dtype: int64


### Week-5 validation design — before the audit

To establish a before/after comparison, I first reproduce the validation design used in Week 5.

The Week-5 Random Forest was trained using observations from October 2025 through January 2026 and evaluated on February 2026.

The selected model configuration was:

- 300 trees
- maximum depth of 10
- minimum samples per leaf of 1
- square-root feature selection

The evaluation metric is Precision@50, matching the Week-5 decision setting where the highest-ranked 50 candidates represent the review capacity.

This result serves as the baseline against which the honest time-aware evaluation will be compared.

In [11]:
# Recreate the Week-5 training period.
#
# Week 5 used October 2025 through January 2026 for model
# development and February 2026 as the evaluation period.
X_train_before = target_data[
    target_data["month"] < pd.Timestamp("2026-02-01")
][feature_cols].copy()

y_train_before = target_data[
    target_data["month"] < pd.Timestamp("2026-02-01")
]["target"].copy()

X_test_before = final_holdout_df[feature_cols].copy()
y_test_before = final_holdout_df["target"].copy()

print("Week-5 training observations:", len(X_train_before))
print("Week-5 evaluation observations:", len(X_test_before))

Week-5 training observations: 17653
Week-5 evaluation observations: 5735


In [12]:
# Week 5 baseline model
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

# Train the model only on the historical training observations.
# February 2026 remains completely unseen during training.

rf_model.fit(X_train_before, y_train_before)

print("Random Forest training complete.")

Random Forest training complete.


In [13]:
# Generate probability scores for the February 2026 observations.
# The positive-class probability represents the model's estimate
# that a candidate will recover in the following month.
week5_scores = rf_model.predict_proba(
    X_test_before
)[:, 1]

# Copy the evaluation data so that the predictions can be
# ranked without changing the original dataset.
week5_results = final_holdout_df.copy()

week5_results["model_score"] = week5_scores

# Rank candidates from highest predicted recovery probability
# to lowest.
week5_results = week5_results.sort_values(
    "model_score",
    ascending=False
)

# Precision@50 measures the proportion of actual recoveries
# among the 50 highest-ranked candidates.
week5_top_50 = week5_results.head(50)

week5_precision_at_50 = (
    week5_top_50["target"].mean()
)

print(
    "Week-5 Precision@50:",
    round(week5_precision_at_50, 4)
)

Week-5 Precision@50: 0.58


In [14]:
# Recreate the Random Forest configuration selected in Week 5.
week5_tuned_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

# Fit the model on the Week-5 training period.
week5_tuned_rf.fit(
    X_train_before,
    y_train_before
)

print("Week-5 Random Forest training completed.")

Week-5 Random Forest training completed.


In [15]:
# Generate probability scores for the February 2026 observations.
# The positive-class probability represents the model's estimate
# that a candidate will recover in the following month.
week5_scores = week5_tuned_rf.predict_proba(
    X_test_before
)[:, 1]

# Copy the evaluation data so that the predictions can be
# ranked without changing the original dataset.
week5_results = final_holdout_df.copy()

week5_results["model_score"] = week5_scores

# Rank candidates from highest predicted recovery probability
# to lowest.
week5_results = week5_results.sort_values(
    "model_score",
    ascending=False
)

# Precision@50 measures the proportion of actual recoveries
# among the 50 highest-ranked candidates.
week5_top_50 = week5_results.head(50)

week5_precision_at_50 = (
    week5_top_50["target"].mean()
)

print(
    "Week-5 Precision@50:",
    round(week5_precision_at_50, 4)
)

Week-5 Precision@50: 0.86


## Honest model selection

To audit the Week-5 validation design, I repeat the same Random Forest model-selection process using a time-aware split.

The same seven hyperparameter configurations and the same five features from Week 5 are used.

The difference is the validation period:

- Week 5: October 2025–January 2026 training, February 2026 validation.
- Week 6: October–December 2025 development, January 2026 validation.

January is used only to select the configuration in this stage. February remains untouched until the final evaluation.

This keeps the modeling procedure consistent while making the validation process time-aware.

### Honest baseline model

The Week-5 baseline Random Forest used 300 trees with the remaining Random Forest parameters left at their default values.

I reproduce the same baseline configuration under the time-aware split. The model is trained on October–December 2025 and evaluated on January 2026.

This provides a baseline comparison under the revised validation design before applying the Week-5 hyperparameter search.

In [16]:
# Recreate the exact Random Forest baseline configuration used in Week 5.
# No hyperparameter tuning is applied at this stage.

honest_baseline = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

# Train the baseline only on October-December 2025.
# January is kept separate for validation.
honest_baseline.fit(
    X_development,
    y_development
)

# Generate recovery probabilities for the January validation data.
baseline_validation_scores = honest_baseline.predict_proba(
    X_validation
)[:, 1]

# Copy the January observations so that we can attach
# the baseline scores without modifying the original data.
baseline_validation_results = validation_df.copy()

baseline_validation_results["model_score"] = (
    baseline_validation_scores
)

# Rank the observations by predicted probability of recovery.
baseline_validation_results = (
    baseline_validation_results
    .sort_values(
        "model_score",
        ascending=False
    )
)

# Select the 50 highest-ranked observations because
# Precision@50 is the evaluation metric used in Week 5.
baseline_top_50 = baseline_validation_results.head(50)

# Calculate the baseline Precision@50 on January.
honest_baseline_precision_at_50 = (
    baseline_top_50["target"].mean()
)

print(
    "Honest baseline January Precision@50:",
    round(honest_baseline_precision_at_50, 4)
)

Honest baseline January Precision@50: 0.42


### Honest tuned model

I repeat the Week-5 Random Forest hyperparameter search using the time-aware development and validation split.

The same seven configurations, five features, and Precision@50 metric are retained.

The difference is that the models are trained on October–December 2025 and selected using January 2026. February 2026 is not used during model selection and will remain as the final holdout period.

In [17]:
# Define the same seven Random Forest configurations used in Week 5.
# We do not add new configurations so that the before/after
# comparison uses the same model search space.

rf_configs = [
    {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 10,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 15,
        "min_samples_leaf": 2,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 20,
        "min_samples_leaf": 2,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 15,
        "min_samples_leaf": 5,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 20,
        "min_samples_leaf": 5,
        "max_features": "sqrt"
    }
]

# Store the January validation result for each configuration.
honest_tuning_results = []

# Repeat the Week-5 model-selection procedure.
# The only change is that January is now the validation period.

for i, config in enumerate(rf_configs, start=1):

    # Create the Random Forest using the current configuration.
    model = RandomForestClassifier(
        **config,
        random_state=42,
        n_jobs=-1
    )

    # Train only on October-December 2025.
    model.fit(
        X_development,
        y_development
    )

    # Generate probability scores for January 2026.
    validation_scores = model.predict_proba(
        X_validation
    )[:, 1]

    # Copy the January validation observations so that
    # we can attach and rank their model scores.
    ranked_validation = validation_df.copy()

    ranked_validation["model_score"] = validation_scores

    # Rank candidates by predicted probability of recovery.
    ranked_validation = ranked_validation.sort_values(
        "model_score",
        ascending=False
    )

    # Evaluate the same top-50 review capacity used in Week 5.
    top_50 = ranked_validation.head(50)

    # Calculate Precision@50.
    precision_at_50 = top_50["target"].mean()

    # Store the configuration and its January result.
    honest_tuning_results.append({
        "configuration": i,
        **config,
        "precision_at_50": precision_at_50
    })

# Convert the results into a DataFrame for comparison.
honest_tuning_results = pd.DataFrame(
    honest_tuning_results
)

# Sort the configurations from best to worst based on
# January Precision@50.
honest_tuning_results = (
    honest_tuning_results
    .sort_values(
        "precision_at_50",
        ascending=False
    )
    .reset_index(drop=True)
)

honest_tuning_results

,configuration,n_estimators,max_depth,min_samples_leaf,max_features,precision_at_50
0,7,300,20.0,5,sqrt,0.88
1,6,300,15.0,5,sqrt,0.84
2,4,300,15.0,2,sqrt,0.82
3,5,300,20.0,2,sqrt,0.80
4,3,300,10.0,1,sqrt,0.72
5,1,200,NaN,1,sqrt,0.44
6,2,300,NaN,1,sqrt,0.42


### Refit the selected honest model

The January validation period selected the Random Forest configuration with 300 trees, a maximum depth of 20, and a minimum leaf size of 5.

Because January was used for model selection, the selected configuration can now be refit using all historical development data available before the final February holdout.

February remains untouched and will be used only for the final out-of-time evaluation.

In [18]:
# Recreate the Random Forest configuration selected using
# January 2026 as the honest validation period.
# No further hyperparameter tuning is performed.

honest_tuned_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

# Refit the selected configuration using all data available
# before the February final holdout.
# This includes October 2025 through January 2026.

honest_tuned_rf.fit(
    pd.concat([
        X_development,
        X_validation
    ]),
    pd.concat([
        y_development,
        y_validation
    ])
)

print(
    "Honest tuned model refitted on October-January data."
)

Honest tuned model refitted on October-January data.


### Final February holdout evaluation

The model configuration was selected using January 2026, and the selected configuration was then refit using all data available through January.

February 2026 has not been used for model selection or tuning. It is therefore treated as the final out-of-time holdout.

Precision@50 is measured using the same top-50 review capacity used throughout the Week-5 analysis.

In [19]:
# Generate recovery probabilities for the February 2026
# final holdout observations.
final_scores = honest_tuned_rf.predict_proba(
    X_final
)[:, 1]

# Copy the February holdout observations so that we can
# attach model scores without modifying the original data.
final_results = final_holdout_df.copy()

final_results["model_score"] = final_scores

# Rank candidates from highest to lowest predicted
# probability of recovery.
final_results = final_results.sort_values(
    "model_score",
    ascending=False
)

# Select the 50 highest-ranked candidates because the
# decision setting uses a review capacity of 50.
final_top_50 = final_results.head(50)

# Calculate Precision@50 on the untouched February holdout.
honest_precision_at_50 = (
    final_top_50["target"].mean()
)

print(
    "Honest final February Precision@50:",
    round(honest_precision_at_50, 4)
)

Honest final February Precision@50: 0.8


### Before/after comparison

The Week-5 evaluation produced an observed Precision@50 of 0.86 using the original validation design, where February 2026 was used for model selection and evaluation.

For the honest time-aware evaluation, the same seven Random Forest configurations were evaluated using January 2026 as the validation period. The best configuration was selected using January and then refit using October 2025 through January 2026. February 2026 was reserved as the final holdout.

| Evaluation | Validation / selection design | Final evaluation period | Precision@50 |
|---|---|---|---:|
| Week 5 — Before | October 2025–January 2026 → February 2026 | February 2026 | **0.86** |
| Week 6 — After | October–December 2025 → January 2026; refit through January | February 2026 | **0.80** |
| Difference | — | — | **−0.06** |

The measured Precision@50 decreased from 0.86 to 0.80 under the time-aware validation design. The selected hyperparameters also changed between the two validation designs.

This suggests that the Week-5 result was sensitive to the validation design. The honest result provides a more conservative estimate of the model's observed ranking performance on an out-of-time holdout.

The result should be treated as decision-support evidence rather than evidence that the model will achieve the same performance on future clients or future periods.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final Random Forest used five features:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_engaged_sessions`

I audit each feature against the prediction timeline and target construction.

The prediction is made using information from the current month. The target is defined using the following month's CTR relative to its position-peer median.

A feature would be considered potentially leaky if it used information from the future outcome month, directly contained the target, or was calculated using information that would not have been available when the prediction was made.

The audit therefore checks:

1. Whether each feature is available in the prediction month.
2. Whether any feature directly uses the next month's outcome.
3. Whether any feature was used to construct the target.
4. Whether the feature aggregation crosses the prediction/outcome boundary.

The audit is based on the actual feature construction and target construction used in this notebook.

First audit: Check whether each feature is available in the prediction month.

In [24]:
# Define the five features used by the Random Forest.
# These are the same features used in the Week-5 model.

final_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

# Create a feature-level leakage audit based on how the
# features were constructed in this notebook.

leakage_audit = pd.DataFrame({
    "feature": final_features,
    "prediction_period": "Current month",
    "future_outcome_used": [
        False,
        False,
        False,
        False,
        False
    ],
    "used_to_construct_target": [
        False,
        False,
        False,
        False,
        False
    ],
    "leakage_status": [
        "No direct leakage identified",
        "No direct leakage identified",
        "No direct leakage identified",
        "No direct leakage identified",
        "No direct leakage identified"
    ]
})

# Display the audit table.
leakage_audit

,feature,prediction_period,future_outcome_used,used_to_construct_target,leakage_status
0,gsc_impressions,Current month,False,False,No direct leakage identified
1,gsc_clicks,Current month,False,False,No direct leakage identified
2,gsc_avg_position,Current month,False,False,No direct leakage identified
3,ga4_pageviews,Current month,False,False,No direct leakage identified
4,ga4_engaged_sessions,Current month,False,False,No direct leakage identified


Second audit: Check whether any feature directly uses the next month's outcome.

In [25]:
# Check the months represented in the training features.
# The model should use current-month observations rather than
# future-month observations.

print("Training feature months:")
print(
    pd.to_datetime(training_data["month"])
    .sort_values()
    .unique()
)

# Check the months represented in the final holdout features.
# These should represent February 2026, the final prediction period.

print("\nFinal holdout feature months:")
print(
    pd.to_datetime(final_holdout_df["month"])
    .sort_values()
    .unique()
)

Training feature months:
<DatetimeArray>
['2025-10-01 00:00:00', '2025-11-01 00:00:00', '2025-12-01 00:00:00',
 '2026-01-01 00:00:00', '2026-02-01 00:00:00']
Length: 5, dtype: datetime64[us]

Final holdout feature months:
<DatetimeArray>
['2026-02-01 00:00:00']
Length: 1, dtype: datetime64[us]


Third audit: Check whether any feature was used to construct the target.

In [26]:
# Confirm which columns contain the future outcome information.
# These columns are intentionally created from the following month
# and therefore must not be included in the model feature set.

future_columns = [
    "next_month",
    "next_ctr",
    "next_peer_median_ctr",
    "next_peer_count",
    "future_ctr_improved",
    "target"
]

print("Future/target columns:")
print(future_columns)

print("\nModel features:")
print(final_features)

Future/target columns:
['next_month', 'next_ctr', 'next_peer_median_ctr', 'next_peer_count', 'future_ctr_improved', 'target']

Model features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


### Leakage audit findings

The five model features were checked against the target construction and prediction timeline.

| Feature | Uses future outcome? | Used to construct target? | Audit finding |
|---|---|---|---|
| `gsc_impressions` | No | No | No direct leakage identified |
| `gsc_clicks` | No | No | No direct leakage identified |
| `gsc_avg_position` | No | No | No direct leakage identified |
| `ga4_pageviews` | No | No | No direct leakage identified |
| `ga4_engaged_sessions` | No | No | No direct leakage identified |

The target is intentionally constructed using next-month information: `next_ctr` and `next_peer_median_ctr`. This information is used to define the outcome that the model is trying to predict and is not included among the five model features.

The model features represent current-month search and engagement signals, while the target represents the following month's outcome. Therefore, based on the feature construction reviewed here, I found no direct target leakage in the final feature set.

The audit also confirms that the February 2026 final holdout contains February feature observations. February was kept separate from model selection and was used only for the final out-of-time evaluation.

This audit does not prove that the dataset is free of every possible form of leakage; it establishes that no direct leakage was identified in the reviewed feature and target construction.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original Week-5 claim

> The tuned Random Forest measured the highest Precision@50 on this evaluation window.

The Week-5 notebook also stated that the result was directional evidence for the evaluation period and should not be interpreted as proof of performance on future data.

### Evidence from the Week-6 audit

Under the Week-5 evaluation design, the tuned Random Forest measured a Precision@50 of 0.86 on February 2026.

Under the time-aware validation design, hyperparameters were selected using January 2026 and February 2026 was reserved as a final holdout. The resulting model measured a Precision@50 of 0.80 on February 2026.

| Evaluation design | Precision@50 |
|---|---:|
| Week 5 — original design | **0.86** |
| Week 6 — time-aware design | **0.80** |

### Rewritten claim

> The tuned Random Forest showed the highest observed Precision@50 among the evaluated approaches under the Week-5 evaluation design. Under the time-aware validation design, the model measured 0.80 Precision@50 on the February 2026 holdout. The results provide directional evidence that the model may support content-review prioritization, but do not establish that the measured performance will generalize to future periods or clients.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.